# 🛡️ Network Intrusion Detection System (NIDS)
## Capstone Project - CIC-IDS-2017 Analysis & XAI

This notebook documents the training, evaluation, and explainability phase of the NIDS project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import shap

# Import project modules
from data_handler import load_cicids_data, preprocess_data, encode_labels
import config

## 1. Data Acquisition & Preprocessing
We use the **CIC-IDS-2017** dataset, which contains 78 features describing network flows.

In [ ]:
# Load data using the project's data handler
DATA_PATH = 'data/merged_network_intrusion_dataset.csv' # Update with your path
if os.path.exists(DATA_PATH):
    df = load_cicids_data(DATA_PATH)
    X, y = preprocess_data(df)
    y_encoded, le = encode_labels(y)
    
    print(f"Dataset loaded: {X.shape[0]} rows, {X.shape[1]} features")
    print("Classes detected:", le.classes_)
else:
    print("Please ensure the dataset is in the data/ folder.")

## 2. Model Training
We train both Random Forest and XGBoost classifiers.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print("Training XGBoost...")
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)

# Save the best model
best_model_path = 'models/best_model.pkl'
os.makedirs('models', exist_ok=True)
with open(best_model_path, 'wb') as f:
    pickle.dump(xgb, f)
print("✅ Best model (XGBoost) saved.")

## 3. Explainable AI (SHAP)
Visualizing feature contributions for a single flow.

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer(X_test[:100])

print("SHAP Waterfall Plot for the first test instance:")
shap.plots.waterfall(shap_values[0])

## 4. Evaluation Metrics
Accuracy, Confusion Matrix, and ROC Curves.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = xgb.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()